# Notebook 4 — DRE com Hierarquia

**Objetivo:** Extrair a DRE da Silver consolidada, reconstruir a hierarquia pai-filho respeitando a cascata de resultados, validar matematicamente e escrever a tabela curada.  
**Input:** `layer_02_silver.n0_dfp_cia_aberta`  
**Output:** `layer_02_silver.n1_dfp_cia_aberta_dre`

## 0. Configuração — Bibliotecas e Conexão com o Banco

In [2]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{quote_plus(os.getenv('DB_USER', 'postgres'))}:"
    f"{quote_plus(os.getenv('DB_PASS', 'password'))}@"
    f"{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', '5432')}/"
    f"{os.getenv('DB_NAME', 'data_lake')}"
)

print('✅ Conexão com o banco configurada.')

✅ Conexão com o banco configurada.


## 1. Extração dos Dados — query com deduplicação e IS_LEAF

> A query extrai os dados da DRE de `n0_dfp_cia_aberta` com três preocupações simultâneas:  
> deduplicação de versões, filtro de empresas elegíveis e seleção das colunas do Golden Schema.

In [3]:
query_dre_hierarquias = '''
WITH base_inicial as (
    SELECT  
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 1) as "CHAVE_BUSCA_NIVEL_1",
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 3) as "CHAVE_BUSCA_NIVEL_2",
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 5) as "CHAVE_BUSCA_NIVEL_3",
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 7) as "CHAVE_BUSCA_NIVEL_4",
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 9) as "CHAVE_BUSCA_NIVEL_5",
        
        -- DEDUPLICAÇÃO NA BASE (Preventiva)
        -- Trazemos apenas uma linha por conta/data/cia (pegando a maior versão ou valor se houver duplicação na origem)
        t.* FROM (
        SELECT DISTINCT ON ("CNPJ_CIA", "DT_REFER", "CD_CONTA") 
            "CNPJ_CIA",
            "DT_REFER",
            "DT_REFER_TRATADO", 
            "DT_REFER_ANO", 
            "VERSAO", 
            "DENOM_CIA", 
            "CD_CVM",  
            "GRUPO_DFP_TRATADO", 
            "DT_FIM_EXERC_TRATADO", 
            "DT_FIM_EXERC_ANO", 
            "CD_CONTA", 
            "CD_CONTA_QTD_DIGITOS", 
            "DS_CONTA", 
            "VL_CONTA_TRATADO", 
            "ST_CONTA_FIXA", 
            "IS_LEAF"
        FROM layer_02_silver.n0_dfp_cia_aberta
        WHERE "GRUPO_DFP_TRATADO" = 'DRE'
        ORDER BY "CNPJ_CIA", "DT_REFER", "CD_CONTA", "VERSAO" DESC
    ) t
),

-- Tabelas de Referência BLINDADAS (Garante 1 linha por Chave)
ref_nivel_1 as (
    SELECT DISTINCT ON ("CHAVE_MATCH")
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 1) as "CHAVE_MATCH",
        UPPER(TRIM("DS_CONTA")) as "DS_NIVEL_1"
    FROM layer_02_silver.n0_dfp_cia_aberta
    WHERE "GRUPO_DFP_TRATADO" = 'DRE' AND "CD_CONTA_QTD_DIGITOS" = 1
    ORDER BY "CHAVE_MATCH", "VERSAO" DESC
),
ref_nivel_2 as (
    SELECT DISTINCT ON ("CHAVE_MATCH")
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 3) as "CHAVE_MATCH",
        UPPER(TRIM("DS_CONTA")) as "DS_NIVEL_2"
    FROM layer_02_silver.n0_dfp_cia_aberta
    WHERE "GRUPO_DFP_TRATADO" = 'DRE' AND "CD_CONTA_QTD_DIGITOS" = 3
    ORDER BY "CHAVE_MATCH", "VERSAO" DESC
),
ref_nivel_3 as (
    SELECT DISTINCT ON ("CHAVE_MATCH")
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 5) as "CHAVE_MATCH",
        UPPER(TRIM("DS_CONTA")) as "DS_NIVEL_3"
    FROM layer_02_silver.n0_dfp_cia_aberta
    WHERE "GRUPO_DFP_TRATADO" = 'DRE' AND "CD_CONTA_QTD_DIGITOS" = 5
    ORDER BY "CHAVE_MATCH", "VERSAO" DESC
),
ref_nivel_4 as (
    SELECT DISTINCT ON ("CHAVE_MATCH")
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 7) as "CHAVE_MATCH",
        UPPER(TRIM("DS_CONTA")) as "DS_NIVEL_4"
    FROM layer_02_silver.n0_dfp_cia_aberta
    WHERE "GRUPO_DFP_TRATADO" = 'DRE' AND "CD_CONTA_QTD_DIGITOS" = 7
    ORDER BY "CHAVE_MATCH", "VERSAO" DESC
),
ref_nivel_5 as (
    SELECT DISTINCT ON ("CHAVE_MATCH")
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 9) as "CHAVE_MATCH",
        UPPER(TRIM("DS_CONTA")) as "DS_NIVEL_5"
    FROM layer_02_silver.n0_dfp_cia_aberta
    WHERE "GRUPO_DFP_TRATADO" = 'DRE' AND "CD_CONTA_QTD_DIGITOS" = 9
    ORDER BY "CHAVE_MATCH", "VERSAO" DESC
)

-- JOIN FINAL (Mantido igual, mas agora seguro)
SELECT 
    base."CNPJ_CIA",
    empresas_selecionadas."SETOR_ATIV",
    base."DT_REFER",
    base."DT_REFER_TRATADO", 
    base."DT_REFER_ANO", 
    base."VERSAO", 
    base."DENOM_CIA", 
    base."CD_CVM",  
    base."GRUPO_DFP_TRATADO", 
    base."DT_FIM_EXERC_TRATADO", 
    base."DT_FIM_EXERC_ANO", 
    base."CD_CONTA", 
    base."CD_CONTA_QTD_DIGITOS", 
    base."DS_CONTA",
    base."CD_CONTA" || ' - ' || base."DS_CONTA" as "CONTA_NOME_COMPLETO",
    base."VL_CONTA_TRATADO", 
    base."ST_CONTA_FIXA", 
    base."IS_LEAF",
    
    COALESCE(n1."DS_NIVEL_1", 'DEMONSTRAÇÃO DO RESULTADO') as "DS_NIVEL_1",
    n2."DS_NIVEL_2",
    n3."DS_NIVEL_3",
    n4."DS_NIVEL_4",
    n5."DS_NIVEL_5",
    
    'layer_02_silver.n0_dfp_cia_aberta' as "_origem_tabela"
FROM base_inicial base
LEFT JOIN ref_nivel_1 n1 ON base."CHAVE_BUSCA_NIVEL_1" = n1."CHAVE_MATCH"
LEFT JOIN ref_nivel_2 n2 ON base."CHAVE_BUSCA_NIVEL_2" = n2."CHAVE_MATCH"
LEFT JOIN ref_nivel_3 n3 ON base."CHAVE_BUSCA_NIVEL_3" = n3."CHAVE_MATCH"
LEFT JOIN ref_nivel_4 n4 ON base."CHAVE_BUSCA_NIVEL_4" = n4."CHAVE_MATCH"
LEFT JOIN ref_nivel_5 n5 ON base."CHAVE_BUSCA_NIVEL_5" = n5."CHAVE_MATCH"
LEFT JOIN layer_02_silver.n0_empresas_selecionadas as empresas_selecionadas
            ON base."CNPJ_CIA" = empresas_selecionadas."CNPJ_CIA"
ORDER BY base."CNPJ_CIA", base."DT_REFER", base."CD_CONTA";
'''

In [4]:
# Executa a query e carrega em memória
with engine.connect() as conn:
    df_dre = pd.read_sql(text(query_dre_hierarquias), con=conn)

print(f'✅ Dados carregados: {df_dre.shape[0]:,} linhas | {df_dre.shape[1]} colunas')
df_dre.info()

✅ Dados carregados: 55,113 linhas | 24 colunas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55113 entries, 0 to 55112
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   CNPJ_CIA              55113 non-null  object 
 1   SETOR_ATIV            55113 non-null  object 
 2   DT_REFER              55113 non-null  object 
 3   DT_REFER_TRATADO      55113 non-null  object 
 4   DT_REFER_ANO          55113 non-null  float64
 5   VERSAO                55113 non-null  object 
 6   DENOM_CIA             55113 non-null  object 
 7   CD_CVM                55113 non-null  object 
 8   GRUPO_DFP_TRATADO     55113 non-null  object 
 9   DT_FIM_EXERC_TRATADO  55113 non-null  object 
 10  DT_FIM_EXERC_ANO      55113 non-null  float64
 11  CD_CONTA              55113 non-null  object 
 12  CD_CONTA_QTD_DIGITOS  55113 non-null  int64  
 13  DS_CONTA              55113 non-null  object 
 14  CONTA_NOME_COMPLETO   5

## 2. Reconstrução Hierárquica — Safe Mode V11

> A CVM permite que empresas reportem apenas contas analíticas, omitindo os pais.  
> Esta etapa reconstrói os pais ausentes somando os filhos — mas **nunca sobrescreve** um valor original (`abs(valor) > 0.01`).

In [5]:
def executar_sanity_check_v11_optimized(df_input):
    print("🏗️ INICIANDO SANITY CHECK V11 (OPTIMIZED & SILVER READY)...\n")
    
    # ==============================================================================
    # 1. DEDUPLICAÇÃO E LIMPEZA INICIAL
    # ==============================================================================
    # Garante chave única (CNPJ + DATA + CONTA).
    # Se houver duplicata de linha, soma os valores.
    cols_group = ['CNPJ_CIA', 'DT_REFER', 'CD_CONTA', 'CD_CONTA_QTD_DIGITOS']
    
    # Agrupamos pelas chaves essenciais. 
    # Usamos 'first' para preservar metadados não-numéricos como DENOM_CIA.
    df = df_input.groupby(cols_group).agg({
        'VL_CONTA_TRATADO': 'sum',
        'DENOM_CIA': 'first' 
    }).reset_index()
    
    print(f"   -> Base inicial limpa e deduplicada: {len(df)} linhas.")
    
    # ==============================================================================
    # 2. FASE DE RECONSTRUÇÃO (BOTTOM-UP WATERFALL)
    # ==============================================================================
    # Ordena níveis do mais profundo para o mais raso (Ex: 7 -> 5 -> 3)
    niveis = sorted(df['CD_CONTA_QTD_DIGITOS'].unique(), reverse=True)
    print(f"   -> Níveis hierárquicos detectados: {niveis}")
    
    total_reconstruido = 0
    
    for nivel_filho in niveis:
        # --- OTIMIZAÇÃO CRÍTICA ---
        # Se estamos no nível 3 (Ex: 3.01), o pai seria Nível 1 (Conta "3").
        # A conta "3" é apenas um título, não tem valor analítico para DRE.
        # Paramos o loop aqui para economizar processamento.
        if nivel_filho <= 3: 
            continue 
        
        # Filtra apenas as contas filhas desse nível
        df_filhos = df[df['CD_CONTA_QTD_DIGITOS'] == nivel_filho].copy()
        if df_filhos.empty: continue

        # Identifica Pai Virtual (Remove tudo após o último ponto)
        # Ex: "3.02.01" -> "3.02"
        df_filhos['CD_PAI_VIRTUAL'] = df_filhos['CD_CONTA'].astype(str).str.rsplit('.', n=1).str[0]
        
        # Calcula a SOMA DOS FILHOS
        # Agrupamos apenas pelas chaves lógicas (sem DENOM_CIA no índice para evitar desalinhamento)
        soma_filhos = df_filhos.groupby(['CNPJ_CIA', 'DT_REFER', 'CD_PAI_VIRTUAL']).agg({
            'VL_CONTA_TRATADO': 'sum',
            'DENOM_CIA': 'first' # Recupera o nome da empresa de qualquer filho
        }).reset_index()
        
        # Prepara o DataFrame calculado para o Merge
        soma_filhos.rename(columns={'VL_CONTA_TRATADO': 'VL_CALCULADO', 'CD_PAI_VIRTUAL': 'CD_CONTA'}, inplace=True)
        soma_filhos['CD_CONTA_QTD_DIGITOS'] = nivel_filho - 2 # Define o nível do pai (Ex: 7 vira 5)
        
        # --- MERGE / UPSERT (USANDO ÍNDICES ALINHADOS) ---
        keys = ['CNPJ_CIA', 'DT_REFER', 'CD_CONTA']
        
        df.set_index(keys, inplace=True)
        soma_filhos.set_index(keys, inplace=True)
        
        # A. UPDATE: Identifica pais que já existem
        idx_comum = df.index.intersection(soma_filhos.index)
        
        if not idx_comum.empty:
            # Lógica de "Force Truth": Confiamos na soma dos filhos.
            vals_calc = soma_filhos.loc[idx_comum, 'VL_CALCULADO']
            vals_orig = df.loc[idx_comum, 'VL_CONTA_TRATADO']
            
            # Atualiza SE: Calculado não for zero E Diferente do Original (com tolerância)
            # Corrige casos como ODONTOPREV (Valor parcial incorreto no pai)
            mask_diff = (vals_calc != 0) & (~np.isclose(vals_calc, vals_orig, atol=1.0))
            idx_update = idx_comum[mask_diff]
            
            if not idx_update.empty:
                df.loc[idx_update, 'VL_CONTA_TRATADO'] = vals_calc.loc[idx_update]
                total_reconstruido += len(idx_update)

        # B. INSERT: Identifica pais que não existem (Linhas Fantasmas)
        idx_novos = soma_filhos.index.difference(df.index)
        
        if not idx_novos.empty:
            # Corrige casos como CESP (Conta 3.02 não existia)
            novos = soma_filhos.loc[idx_novos].reset_index()
            novos.rename(columns={'VL_CALCULADO': 'VL_CONTA_TRATADO'}, inplace=True)
            
            df.reset_index(inplace=True)
            df = pd.concat([df, novos], ignore_index=True)
            total_reconstruido += len(idx_novos)
        else:
            df.reset_index(inplace=True)

    print(f"✅ Reconstrução concluída. Total de intervenções nos dados: {total_reconstruido}\n")
    
    # ==============================================================================
    # 3. FASE DE VALIDAÇÃO (SANITY CHECK FINAL)
    # ==============================================================================
    print("2️⃣ FASE DE VALIDAÇÃO (Matemática Contábil)...")
    
    df_pivot = df.pivot_table(
        index=['CNPJ_CIA', 'DENOM_CIA', 'DT_REFER'],
        columns='CD_CONTA',
        values='VL_CONTA_TRATADO',
        aggfunc='sum'
    ).fillna(0)
    
    TOLERANCIA = 1.0

    # TESTE 1: Lucro Bruto (3.03 = 3.01 + 3.02)
    # Verifica tanto a soma direta (padrão) quanto a subtração do absoluto (caso sinal trocado)
    if all(col in df_pivot.columns for col in ['3.01', '3.02', '3.03']):
        soma_direta = df_pivot['3.01'] + df_pivot['3.02']
        soma_abs = df_pivot['3.01'] - df_pivot['3.02'].abs()
        
        match = ((df_pivot['3.03'] - soma_direta).abs() <= TOLERANCIA) | \
                ((df_pivot['3.03'] - soma_abs).abs() <= TOLERANCIA)
        
        falhas = df_pivot[~match]
        print(f"   -> TESTE 1 (Res. Bruto): {len(falhas)} falhas detectadas.")
        if not falhas.empty:
            # Print de Debug apenas se houver falha
            print(falhas[['3.01', '3.02', '3.03']].head(3))

    # TESTE 2: EBIT (3.05 = 3.03 + 3.04)
    if all(col in df_pivot.columns for col in ['3.03', '3.04', '3.05']):
        soma_direta = df_pivot['3.03'] + df_pivot['3.04']
        match = (df_pivot['3.05'] - soma_direta).abs() <= TOLERANCIA
        
        falhas = df_pivot[~match]
        print(f"   -> TESTE 2 (EBIT): {len(falhas)} falhas detectadas.")
        if not falhas.empty:
            falhas['DIFF'] = falhas['3.05'] - (falhas['3.03'] + falhas['3.04'])
            print(falhas[['3.03', '3.04', '3.05', 'DIFF']].head(3))
            
    return df # Retorna o DataFrame pronto para ser salvo na Silver

## 3. Padronização e Rastreabilidade — Golden Schema V13

> Após a reconstrução numérica, esta etapa adiciona as colunas de qualidade:  
> `FLAG_RECONSTRUCAO`, `FLAG_NORMALIZACAO`, `DS_CONTA_REPORTADA` e `STATUS_MATH`.

In [6]:
def finalizar_dre_silver_traceability_v13_fix(df_curado, df_original_completo):
    print("💎 INICIANDO FINALIZAÇÃO V13.1 (TRACEABILITY FIX)...\n")
    
    # Cópia para trabalho
    df_final = df_curado.copy()
    
    # Definição do Schema Final
    COLUNAS_ALVO = [
        "CNPJ_CIA", "SETOR_ATIV", "DT_REFER", "DT_REFER_TRATADO", "DT_REFER_ANO", 
        "VERSAO", "DENOM_CIA", "CD_CVM", "GRUPO_DFP_TRATADO", 
        "DT_FIM_EXERC_TRATADO", "DT_FIM_EXERC_ANO", 
        "CD_CONTA", "CD_CONTA_QTD_DIGITOS", 
        "DS_CONTA",                 # Nome Final (Padronizado)
        "DS_CONTA_REPORTADA",       # Nome Original (Raw)
        "FLAG_NORMALIZACAO",        # Booleano
        "FLAG_RECONSTRUCAO",        # Booleano
        "CONTA_NOME_COMPLETO", 
        "VL_CONTA_TRATADO", 
        "ST_CONTA_FIXA",            # Flag Final
        "ST_CONTA_FIXA_REPORTADA",  # Flag Original
        "IS_LEAF", 
        "DS_NIVEL_1", "DS_NIVEL_2", "DS_NIVEL_3", "DS_NIVEL_4", "DS_NIVEL_5", 
        "_origem_tabela"
    ]
    
    # ==========================================================================
    # 1. RESGATE DO DADO ORIGINAL (AUDITORIA) - CORREÇÃO DE RENOMEAÇÃO
    # ==========================================================================
    print("1️⃣ Resgatando dados reportados originalmente...")
    
    # Prepara o original (Maior versão, sem duplicatas de chave)
    df_orig_ref = df_original_completo.sort_values('VERSAO', ascending=False).drop_duplicates(['CNPJ_CIA', 'DT_REFER', 'CD_CONTA']).copy()
    
    # --- CORREÇÃO AQUI: Renomear ANTES do Merge ---
    # Isso garante que a coluna entre com o nome certo, haja colisão ou não
    df_orig_ref.rename(columns={
        'DS_CONTA': 'DS_CONTA_REPORTADA',
        'ST_CONTA_FIXA': 'ST_CONTA_FIXA_REPORTADA'
    }, inplace=True)
    
    # Seleciona apenas o necessário para o join
    cols_origem = ['CNPJ_CIA', 'DT_REFER', 'CD_CONTA', 'DS_CONTA_REPORTADA', 'ST_CONTA_FIXA_REPORTADA']
    
    # Merge Left
    df_final = pd.merge(
        df_final,
        df_orig_ref[cols_origem],
        on=['CNPJ_CIA', 'DT_REFER', 'CD_CONTA'],
        how='left'
    )
    
    # Se DS_CONTA_REPORTADA for NaN, a linha foi criada na V11 (Reconstruída)
    df_final['FLAG_RECONSTRUCAO'] = df_final['DS_CONTA_REPORTADA'].isna()
    
    print(f"   -> Linhas Reconstruídas (Sem original): {df_final['FLAG_RECONSTRUCAO'].sum()}")

    # ==========================================================================
    # 2. GOLDEN MAP (PADRONIZAÇÃO)
    # ==========================================================================
    print("2️⃣ Aplicando Golden Map...")
    
    # Mapa de Nomes (Baseado na frequência histórica)
    df_nomes = df_original_completo[['CD_CONTA', 'DS_CONTA']].dropna()
    df_nomes['DS_CONTA'] = df_nomes['DS_CONTA'].str.strip().str.upper()
    
    ranking = df_nomes.groupby(['CD_CONTA', 'DS_CONTA']).size().reset_index(name='FREQ')
    ranking.sort_values(['CD_CONTA', 'FREQ'], ascending=[True, False], inplace=True)
    
    golden_map = ranking.drop_duplicates(subset=['CD_CONTA']).set_index('CD_CONTA')['DS_CONTA'].to_dict()
    
    # Mapa de Conta Fixa
    df_fixa = df_original_completo[['CD_CONTA', 'ST_CONTA_FIXA']].dropna()
    ranking_fixa = df_fixa.groupby(['CD_CONTA', 'ST_CONTA_FIXA']).size().reset_index(name='FREQ')
    ranking_fixa.sort_values(['CD_CONTA', 'FREQ'], ascending=[True, False], inplace=True)
    golden_map_fixa = ranking_fixa.drop_duplicates(subset=['CD_CONTA']).set_index('CD_CONTA')['ST_CONTA_FIXA'].to_dict()
    
    # Aplicação
    df_final['DS_CONTA'] = df_final['CD_CONTA'].map(golden_map)
    # Fallback: Se não tem no mapa, usa o reportado. Se não tem reportado, gera genérico.
    df_final['DS_CONTA'] = df_final['DS_CONTA'].fillna(df_final['DS_CONTA_REPORTADA']).fillna("CONTA RECONSTRUÍDA - " + df_final['CD_CONTA'])
    
    df_final['ST_CONTA_FIXA'] = df_final['CD_CONTA'].map(golden_map_fixa).fillna('S')

    # ==========================================================================
    # 3. FLAGS DE AUDITORIA
    # ==========================================================================
    nome_final_norm = df_final['DS_CONTA'].str.strip().str.upper()
    nome_orig_norm = df_final['DS_CONTA_REPORTADA'].fillna("").str.strip().str.upper()
    
    # Normalizou se nomes diferentes E não é reconstrução
    df_final['FLAG_NORMALIZACAO'] = (nome_final_norm != nome_orig_norm) & (~df_final['FLAG_RECONSTRUCAO'])
    
    print(f"   -> Linhas Normalizadas (Renomeadas): {df_final['FLAG_NORMALIZACAO'].sum()}")

    # ==========================================================================
    # 4. ENRIQUECIMENTO E HIERARQUIAS
    # ==========================================================================
    print("3️⃣ Gerando Metadados e Hierarquias...")
    
    # Enriquecimento Temporal
    cols_meta = ['CNPJ_CIA', 'DT_REFER', 'VERSAO', 'DT_FIM_EXERC_TRATADO', 'DT_FIM_EXERC_ANO', 'CD_CVM', 'SETOR_ATIV']
    df_meta = df_original_completo[cols_meta].sort_values('VERSAO', ascending=False).drop_duplicates(subset=['CNPJ_CIA', 'DT_REFER'])
    
    # Remove colunas que já existem (exceto chaves) para evitar sufixos no merge
    for col in cols_meta:
        if col in df_final.columns and col not in ['CNPJ_CIA', 'DT_REFER']:
            df_final.drop(columns=[col], inplace=True)
            
    df_final = pd.merge(df_final, df_meta, on=['CNPJ_CIA', 'DT_REFER'], how='left')
    
    # Preenche Gaps
    df_final['SETOR_ATIV'] = df_final.groupby('CNPJ_CIA')['SETOR_ATIV'].ffill().bfill()
    df_final['CD_CVM'] = df_final.groupby('CNPJ_CIA')['CD_CVM'].ffill().bfill()
    df_final['DENOM_CIA'] = df_final.groupby('CNPJ_CIA')['DENOM_CIA'].ffill().bfill()
    
    # IS_LEAF
    df_final['TEMP_PAI'] = df_final['CD_CONTA'].astype(str).str.rsplit('.', n=1).str[0]
    pais_existentes = set(df_final['TEMP_PAI'].unique())
    df_final['IS_LEAF'] = ~df_final['CD_CONTA'].isin(pais_existentes)
    
    # Hierarquias
    golden_map['3'] = "DEMONSTRAÇÃO DO RESULTADO"
    
    for i in range(1, 6):
        if i == 1:
            cod_nivel = df_final['CD_CONTA'].str.split('.').str[0]
        else:
            cod_nivel = df_final['CD_CONTA'].apply(lambda x: ".".join(x.split('.')[:i]) if x.count('.') >= (i-1) else np.nan)
        
        df_final[f'DS_NIVEL_{i}'] = cod_nivel.map(golden_map)

    # ==========================================================================
    # 5. FINALIZAÇÃO
    # ==========================================================================
    df_final['DT_REFER'] = pd.to_datetime(df_final['DT_REFER'])
    df_final['DT_REFER_TRATADO'] = df_final['DT_REFER']
    df_final['DT_REFER_ANO'] = df_final['DT_REFER'].dt.year
    df_final['VERSAO'] = df_final['VERSAO'].fillna(1).astype(int)
    
    df_final['CONTA_NOME_COMPLETO'] = df_final['CD_CONTA'] + ' - ' + df_final['DS_CONTA']
    df_final['_origem_tabela'] = 'layer_02_silver.n0_dfp_cia_aberta (Curated V13.1)'
    
    # Garante todas as colunas
    for col in COLUNAS_ALVO:
        if col not in df_final.columns:
            df_final[col] = None
            
    df_final = df_final[COLUNAS_ALVO]
    
    print(f"✅ Processo Concluído. Schema Final: {df_final.shape}")
    return df_final

## 4. Validação Matemática Final — integridade contábil antes de escrever

> Antes de gravar no banco, validamos se cada par pai-filho está correto (`abs(pai - soma_filhos) ≤ 1000`).  
> A escrita só ocorre se a validação aprovar.

In [7]:
def validar_matematica_final(df_final, etapa_nome="FINAL SILVER"):
    print(f"\n⚖️ EXECUTANDO PROVA REAL NA ETAPA: {etapa_nome}")
    print("-" * 60)
    
    # Verifica se duplicamos linhas no Join (Erro comum em enriquecimento)
    # A chave primária (CNPJ + DATA + CONTA) deve ser única
    duplicatas = df_final.groupby(['CNPJ_CIA', 'DT_REFER', 'CD_CONTA']).size()
    duplicatas = duplicatas[duplicatas > 1]
    
    if not duplicatas.empty:
        print(f"❌ ERRO CRÍTICO: Detectadas {len(duplicatas)} chaves duplicadas após enriquecimento!")
        print("   Isso significa que o Merge (V13) explodiu a cardinalidade.")
        return False
    else:
        print("✅ Integridade de Chaves: OK (Sem duplicatas).")

    # Pivotagem para Teste Matemático
    # Usamos aggfunc='sum' -> se houver duplicata escondida, o valor dobra e o teste falha.
    df_pivot = df_final.pivot_table(
        index=['CNPJ_CIA', 'DENOM_CIA', 'DT_REFER'],
        columns='CD_CONTA',
        values='VL_CONTA_TRATADO',
        aggfunc='sum'
    ).fillna(0)
    
    TOLERANCIA = 1.0
    erros_encontrados = 0

    # --- TESTE 1: LUCRO BRUTO (3.03 = 3.01 + 3.02) ---
    if all(c in df_pivot.columns for c in ['3.01', '3.02', '3.03']):
        # Testa Soma Direta e Soma Absoluta (Sinais trocados)
        soma_direta = df_pivot['3.01'] + df_pivot['3.02']
        soma_abs = df_pivot['3.01'] - df_pivot['3.02'].abs()
        
        match = ((df_pivot['3.03'] - soma_direta).abs() <= TOLERANCIA) | \
                ((df_pivot['3.03'] - soma_abs).abs() <= TOLERANCIA)
        
        falhas = df_pivot[~match]
        if not falhas.empty:
            print(f"❌ FALHA NO LUCRO BRUTO: {len(falhas)} casos.")
            # Debug rápido
            exemplo = falhas.iloc[0]
            print(f"   Ex: {exemplo.name[1]} (3.03 Reportado: {exemplo['3.03']} vs Calc: {exemplo['3.01'] + exemplo['3.02']})")
            erros_encontrados += len(falhas)
        else:
            print("✅ Teste Lucro Bruto (3.03): SUCESSO ABSOLUTO.")
    
    # --- TESTE 2: EBIT (3.05 = 3.03 + 3.04) ---
    if all(c in df_pivot.columns for c in ['3.03', '3.04', '3.05']):
        soma_direta = df_pivot['3.03'] + df_pivot['3.04']
        match = (df_pivot['3.05'] - soma_direta).abs() <= TOLERANCIA
        
        falhas = df_pivot[~match]
        if not falhas.empty:
            print(f"❌ FALHA NO EBIT: {len(falhas)} casos.")
            exemplo = falhas.iloc[0]
            print(f"   Ex: {exemplo.name[1]} (3.05 Reportado: {exemplo['3.05']} vs Calc: {exemplo['3.03'] + exemplo['3.04']})")
            erros_encontrados += len(falhas)
        else:
            print("✅ Teste EBIT (3.05): SUCESSO ABSOLUTO.")

    print("-" * 60)
    if erros_encontrados == 0:
        print("🏆 CERTIFICADO: A tabela está matematicamente perfeita e pronta para o Lake.")
        return True
    else:
        print("⚠️ ALERTA: A tabela tem inconsistências e NÃO deve ser salva ainda.")
        return False

## 5. Normalização do Golden Schema — 29 colunas padronizadas

> Garante que a tabela final tenha exatamente as colunas do Golden Schema V13.5,  
> permitindo `UNION ALL` com BP e DFC sem conflito de schema.

📋 Schema Alvo Final: Tabelas Harmonizadas (Golden Schema)

Abaixo está a definição da estrutura unificada para as tabelas da camada Silver (`BP`, `DRE`, `DFC`). O objetivo deste schema é permitir o empilhamento (`UNION ALL`) e a criação de uma dimensão temporal e de entidade consistentes para Business Intelligence.

| Grupo | Coluna | Tipo | Descrição / Racional |
| :--- | :--- | :--- | :--- |
| **Chaves** | `CNPJ_CIA` | Texto | Chave primária da entidade (CNPJ). |
| | `DT_REFER` | Data | Data de corte do balanço. Fundamental para *Time Intelligence*. |
| | `CD_CONTA` | Texto | Código contábil (Ex: `6.01`). É o ID único da linha no demonstrativo. |
| **Entidade** | `DENOM_CIA` | Texto | Razão social / Nome da empresa. |
| | `CD_CVM` | Int | Código CVM (usado para joins com dados de mercado/ações). |
| | `SETOR_ATIV` | Texto | Setor de atividade econômica na B3. |
| **Tempo** | `DT_REFER_TRATADO` | Date | Versão "Data Pura" (sem hora) do `DT_REFER`. |
| | `DT_REFER_ANO` | Int | Ano (ex: 2021, 2022) para filtros rápidos e particionamento. |
| | `DT_FIM_EXERC_TRATADO`| Date | Data de fim do exercício fiscal (útil para empresas com ano fiscal deslocado). |
| | `DT_FIM_EXERC_ANO` | Int | Ano do exercício fiscal. |
| | `VERSAO` | Int | Versão do documento entregue à CVM (1, 2...). |
| **Negócio** | `GRUPO_DFP_TRATADO` | Texto | Identificador do demonstrativo: `'BP'`, `'DRE'`, `'DFC'`. Chave do empilhamento. |
| | `VL_CONTA_TRATADO` | Float | Valor monetário unificado. Mesma escala e moeda para todas as linhas. |
| | `DS_CONTA` | Texto | Nome da conta **Curado/Padronizado** (Ex: "CAIXA E EQUIVALENTES"). |
| | `CONTA_NOME_COMPLETO` | Texto | Concatenação `CD_CONTA` + `DS_CONTA`. Ideal para eixos de gráficos. |
| | `CD_CONTA_QTD_DIGITOS`| Int | Nível hierárquico numérico da conta (1, 3, 5...). |
| | `ST_CONTA_FIXA` | Texto | `'S'` ou `'N'`. Indica se é uma conta fixa obrigatória da taxonomia CVM. |
| **Qualidade** | `DS_CONTA_REPORTADA` | Texto | Nome original vindo do CSV (Ex: "Variação monetária"). Vital para auditoria (Traceability). |
| **(Traceability)** | `ST_CONTA_FIXA_REPORTADA`| Texto | Status original da conta fixa vindo do arquivo bruto. |
| | `FLAG_NORMALIZACAO` | Bool | `True` se o nome da conta foi alterado/padronizado via *Golden Map*. |
| | `FLAG_RECONSTRUCAO` | Bool | `True` se o valor foi calculado via Python (*Safe Mode*) ou se a linha foi criada artificialmente. |
| | `STATUS_MATH` | Texto | Status da validação contábil: `'OK'`, `'DIVERGENTE'` ou `'DESBALANCEADO'`. |
| **Estrutura** | `IS_LEAF` | Bool | Indica se é o último nível (analítico). Útil para evitar soma duplicada em BI. |
| | `DS_NIVEL_1` ... `5` | Texto | Colunas de hierarquia para *Drill-Down* visual e matrizes. |
| **Metadados** | `_origem_tabela` | Texto | Nome da tabela da camada Bronze de onde o dado foi extraído. |

In [8]:
def padronizar_schema_silver(df_final, demonstrativo):
    """
    Recebe o DataFrame final de qualquer demonstrativo (BP, DRE, DFC) 
    e garante que ele tenha as 26 colunas do Golden Schema.
    """
    # 1. Renomeação de Colunas Específicas para o Padrão
    # Se existir STATUS_DFC (no caso da DFC), vira STATUS_MATH
    if 'STATUS_DFC' in df_final.columns:
        df_final.rename(columns={'STATUS_DFC': 'STATUS_MATH'}, inplace=True)
    
    # Se existir validação do BP (Active - Passive), vira STATUS_MATH
    if 'STATUS_BALANCO' in df_final.columns:
        df_final.rename(columns={'STATUS_BALANCO': 'STATUS_MATH'}, inplace=True)
        
    # Se não tiver STATUS_MATH (ex: DRE que não implementamos checagem horizontal ainda), cria OK
    if 'STATUS_MATH' not in df_final.columns:
        df_final['STATUS_MATH'] = 'NAO_APLICAVEL' # Ou 'OK' se confiar cegamente

    # 2. Garantia de Colunas de Qualidade (Para BP que estava sem)
    cols_qualidade = ['DS_CONTA_REPORTADA', 'ST_CONTA_FIXA_REPORTADA', 'FLAG_NORMALIZACAO', 'FLAG_RECONSTRUCAO']
    for col in cols_qualidade:
        if col not in df_final.columns:
            # Se não tem, é porque não fizemos o merge com o original nesse script.
            # Idealmente, o script do BP deve ser atualizado para fazer o merge igual ao da DFC.
            # Por enquanto, preenchemos com Nulo/False para não quebrar o schema.
            df_final[col] = None 
            if 'FLAG' in col: df_final[col] = False

    # 3. Criação de Colunas Calculadas Faltantes
    if 'CONTA_NOME_COMPLETO' not in df_final.columns:
        df_final['CONTA_NOME_COMPLETO'] = df_final['CD_CONTA'].astype(str) + ' - ' + df_final['DS_CONTA'].astype(str)

    # 4. Seleção e Ordenação Final
    COLUNAS_ALVO = [
        "CNPJ_CIA", "SETOR_ATIV", "DT_REFER", "DT_REFER_TRATADO", "DT_REFER_ANO", 
        "VERSAO", "DENOM_CIA", "CD_CVM", "GRUPO_DFP_TRATADO", 
        "DT_FIM_EXERC_TRATADO", "DT_FIM_EXERC_ANO", 
        "CD_CONTA", "CD_CONTA_QTD_DIGITOS", 
        "DS_CONTA", "DS_CONTA_REPORTADA",       
        "FLAG_NORMALIZACAO", "FLAG_RECONSTRUCAO", "STATUS_MATH",
        "CONTA_NOME_COMPLETO", "VL_CONTA_TRATADO", "ST_CONTA_FIXA", "ST_CONTA_FIXA_REPORTADA",  
        "IS_LEAF", "DS_NIVEL_1", "DS_NIVEL_2", "DS_NIVEL_3", "DS_NIVEL_4", "DS_NIVEL_5", 
        "_origem_tabela"
    ]
    
    # Garante que todas existam
    for col in COLUNAS_ALVO:
        if col not in df_final.columns:
            df_final[col] = None
            
    return df_final[COLUNAS_ALVO]

## 6. Execução do Pipeline Completo

> Encadeia as 4 etapas em sequência: reconstrução → padronização → validação → escrita.

In [9]:
print('=' * 60)
print('PIPELINE DRE — CAMADA SILVER')
print('=' * 60)

# PASSO 1: Reconstrução hierárquica
print('\n>>> PASSO 1: RECONSTRUÇÃO HIERÁRQUICA (V11)')
df_curado = executar_sanity_check_v11_optimized(df_dre)

# PASSO 2: Padronização, enriquecimento e flags de auditoria
print('\n>>> PASSO 2: PADRONIZAÇÃO E RASTREABILIDADE (V13.1)')
df_silver_final = finalizar_dre_silver_traceability_v13_fix(df_curado, df_dre)

# PASSO 3: Normalização para o Golden Schema de 29 colunas
print('\n>>> PASSO 3: NORMALIZAÇÃO DO GOLDEN SCHEMA')
df_dre_ready = padronizar_schema_silver(df_silver_final, 'DRE')

# PASSO 4: Validação matemática final
print('\n>>> PASSO 4: VALIDAÇÃO MATEMÁTICA FINAL')
aprovado = validar_matematica_final(df_dre_ready, 'DRE SILVER FINAL')

print(f'\n📋 Shape final: {df_dre_ready.shape[0]:,} linhas | {df_dre_ready.shape[1]} colunas')

PIPELINE DRE — CAMADA SILVER

>>> PASSO 1: RECONSTRUÇÃO HIERÁRQUICA (V11)
🏗️ INICIANDO SANITY CHECK V11 (OPTIMIZED & SILVER READY)...

   -> Base inicial limpa e deduplicada: 55113 linhas.
   -> Níveis hierárquicos detectados: [np.int64(7), np.int64(5), np.int64(3)]
✅ Reconstrução concluída. Total de intervenções nos dados: 5387

2️⃣ FASE DE VALIDAÇÃO (Matemática Contábil)...
   -> TESTE 1 (Res. Bruto): 0 falhas detectadas.
   -> TESTE 2 (EBIT): 0 falhas detectadas.

>>> PASSO 2: PADRONIZAÇÃO E RASTREABILIDADE (V13.1)
💎 INICIANDO FINALIZAÇÃO V13.1 (TRACEABILITY FIX)...

1️⃣ Resgatando dados reportados originalmente...
   -> Linhas Reconstruídas (Sem original): 5349
2️⃣ Aplicando Golden Map...
   -> Linhas Normalizadas (Renomeadas): 6166
3️⃣ Gerando Metadados e Hierarquias...
✅ Processo Concluído. Schema Final: (60462, 28)

>>> PASSO 3: NORMALIZAÇÃO DO GOLDEN SCHEMA

>>> PASSO 4: VALIDAÇÃO MATEMÁTICA FINAL

⚖️ EXECUTANDO PROVA REAL NA ETAPA: DRE SILVER FINAL
----------------------------

## 7. Escrita na Camada Silver — `n1_dfp_cia_aberta_dre`

> Esta célula **sobrescreve** a tabela. Execute apenas após a validação aprovar.

In [10]:
if aprovado:
    print('💾 Escrevendo tabela no banco...')
    rows = df_dre_ready.to_sql(
        name='n1_dfp_cia_aberta_dre',
        schema='layer_02_silver',
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=10000,
        method='multi'
    )
    print(f'✅ {rows:,} linhas escritas em layer_02_silver.n1_dfp_cia_aberta_dre')
else:
    print('🚨 Pipeline não certificado. Corrija os erros antes de escrever no banco.')

💾 Escrevendo tabela no banco...
✅ 60,462 linhas escritas em layer_02_silver.n1_dfp_cia_aberta_dre
